# 10 - Modelo: Isolation Forest para anomalias em itens de pedido
**Squad 1 — Data Quality em Tempo Real | ML em Streaming**
**Responsável:** Riuler José Silvério
**Problema:** P1 · Anomalias em pedidos e itens · **Modelo A:** não supervisionado clássico
**Comparação:** Modelo B (Autoencoder) de Guilherme & Carlos

### Objetivo
Treinar o Isolation Forest **sem rótulo** na tabela de features (notebook `09`) e avaliar contra o **rótulo fraco**, ou seja, os itens reprovados pelas regras da etapa 1.

### Como funciona
O Isolation Forest monta árvores com cortes aleatórios. Um ponto anômalo fica isolado em poucos cortes e recebe um score baixo. Ele não precisa de rótulo: aprende o que é "comum" e marca o que foge disso. O parâmetro `contamination` define que fração do **treino** vira alerta, e esse limiar é aplicado depois a cada micro-lote novo.

### Versões comparadas (decididas na EDA)
| Versão | Features | Por quê |
|---|---|---|
| **A · principal** | 7 de item e pedido | As 9 de item e pedido do `09`, sem `participacao_item` e `qtd_total_pedido` (ver abaixo) |
| **A · 9 features** | as 9 de item e pedido do `09` | Referência: a versão antes do ajuste |
| **A · sem valor** | a principal sem `log_valor_liquido_item` | O valor líquido sozinho separa o rótulo (força 1,0). Sem ele, dá para ver o que o modelo acha "por conta própria" |
| **A · só item** | preço, quantidade, desconto e valor do item | Mostra quanto o contexto do pedido ajuda ou atrapalha |
| **A · + rastreio** | a principal + 10 de rastreamento | Testa o plano original da planilha (horas entre eventos) |

**Por que `participacao_item` e `qtd_total_pedido` saíram da principal:** na análise dos alertas sem regra (Etapa 6), a `participacao_item` era o motivo de cerca de 70% dos alertas. Ela marcava todo pedido de **1 item só** (participação = 1, algo raro na base) e os pedidos com soma negativa, o que é estrutura do pedido e não anomalia do item. A `qtd_total_pedido` repete a informação de `n_itens_pedido` e da quantidade. No teste local com os dados do SQL, tirar as duas levou o F1 de 0,56 para 0,66 e o PR-AUC de 0,55 para 0,74, e reduziu os alertas sem regra de 165 para 98.

> Como essa escolha olhou para o teste, a métrica do teste da versão principal fica levemente otimista. Na comparação com o Autoencoder, deixe isso registrado.

### Métricas (definidas na planilha)
Precision, Recall, F1 e PR-AUC contra o rótulo fraco, além do % de registros alertados. A métrica principal é o **teste**, com lotes de 08/10 em diante que o modelo não viu no treino.

In [ ]:
# Credenciais via Secret Scope (nunca no código)
adls_client_id = dbutils.secrets.get(scope="internship-squad1", key="adls-client-id")
adls_tenant_id = dbutils.secrets.get(scope="internship-squad1", key="adls-tenant-id")
adls_client_secret = dbutils.secrets.get(scope="internship-squad1", key="adls-client-secret")
adls_storage_account = dbutils.secrets.get(scope="internship-squad1", key="adls-storage-account")

_conta = f"{adls_storage_account}.dfs.core.windows.net"
adls_options = {
    f"fs.azure.account.auth.type.{_conta}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{_conta}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{_conta}": adls_client_id,
    f"fs.azure.account.oauth2.client.secret.{_conta}": adls_client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{_conta}": f"https://login.microsoftonline.com/{adls_tenant_id}/oauth2/token",
}

base_squad1 = f"abfss://squad1@{_conta}"
caminhos = {
    "features_itens": f"{base_squad1}/dupla1/ml/features_itens_pedido",   # entrada (notebook 09)
    "scores_itens": f"{base_squad1}/dupla1/ml/scores_itens_pedido",       # saída deste notebook
}
print("Credenciais e caminhos prontos.")

### Etapa 1 — Leitura das features

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, precision_recall_curve

AZUL, VERMELHO, AMARELO, CINZA = "#1F4E8C", "#C0392B", "#F2B705", "#7F8C8D"
plt.rcParams.update({"figure.figsize": (10, 4), "axes.spines.top": False, "axes.spines.right": False})

df = (spark.read.format("delta").options(**adls_options).load(caminhos["features_itens"])
      .toPandas().sort_values(["data_lote", "lote", "id_item_pedido"]).reset_index(drop=True))

FEAT_ITEM_PEDIDO = ["log_preco", "quantidade", "razao_desconto", "log_valor_liquido_item",
                    "n_itens_pedido", "n_skus_pedido", "qtd_total_pedido", "log_valor_liquido_pedido",
                    "participacao_item"]
FEAT_RASTREIO = ["n_eventos_rastreio", "n_transportadoras", "horas_total_rastreio", "max_horas_entre_eventos",
                 "media_horas_entre_eventos", "qtd_regressoes_status", "teve_extraviado",
                 "status_mais_avancado", "tem_rastreio", "tem_intervalo_rastreio"]
FEAT_PRINCIPAL = [f for f in FEAT_ITEM_PEDIDO if f not in ("participacao_item", "qtd_total_pedido")]
VERSOES = {
    "A · principal": FEAT_PRINCIPAL,
    "A · 9 features": FEAT_ITEM_PEDIDO,
    "A · sem valor": [f for f in FEAT_PRINCIPAL if f != "log_valor_liquido_item"],
    "A · só item": ["log_preco", "quantidade", "razao_desconto", "log_valor_liquido_item"],
    "A · + rastreio": FEAT_PRINCIPAL + FEAT_RASTREIO,
}

treino, teste = df[df.split == "treino"], df[df.split == "teste"]
print(f"Treino: {len(treino):,} itens ({treino.rotulo_fraco.mean():.2%} rotulados) | "
      f"Teste: {len(teste):,} itens ({teste.rotulo_fraco.mean():.2%} rotulados)")

### Etapa 2 — Treino e avaliação das três versões

- `contamination = 0.09`, perto da taxa do rótulo no treino (8,95%). O limiar sai do treino e é aplicado sem ajuste no teste, como seria no streaming.
- `n_estimators = 300` e `random_state = 42`, para ter um resultado reproduzível.
- O **PR-AUC** não depende do limiar: mede o quanto o score ordena os rotulados acima dos demais. Um modelo aleatório teria PR-AUC igual à taxa do rótulo (cerca de 0,08).

In [ ]:
CONTAMINATION = 0.09

def treinar(features, contamination=CONTAMINATION):
    modelo = IsolationForest(n_estimators=300, contamination=contamination, random_state=42, n_jobs=-1)
    modelo.fit(treino[features])
    return modelo

def avaliar(modelo, features, base, nome_base):
    anomalia = -modelo.score_samples(base[features])        # maior = mais anômalo
    alerta = (modelo.predict(base[features]) == -1).astype(int)
    y = base.rotulo_fraco
    return {"base": nome_base, "% alertados": alerta.mean() * 100,
            "Precision": precision_score(y, alerta, zero_division=0),
            "Recall": recall_score(y, alerta, zero_division=0),
            "F1": f1_score(y, alerta, zero_division=0),
            "PR-AUC": average_precision_score(y, anomalia)}

modelos, linhas = {}, []
for nome, feats in VERSOES.items():
    modelos[nome] = treinar(feats)
    for nome_base, base in [("treino", treino), ("teste", teste)]:
        linhas.append({"versão": nome, "nº features": len(feats), **avaliar(modelos[nome], feats, base, nome_base)})

resultado = pd.DataFrame(linhas)
print(f"Referência aleatória de PR-AUC no teste: {teste.rotulo_fraco.mean():.3f}")
display(resultado.round(3))

### Etapa 3 — Curva Precision × Recall no teste

Mostra o que cada versão entregaria em limiares diferentes. Uma curva mais alta e mais à direita é melhor.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for (nome, feats), cor in zip(VERSOES.items(), [AZUL, CINZA, AMARELO, "#2E7D32", "#8E44AD"]):
    p, r, _ = precision_recall_curve(teste.rotulo_fraco, -modelos[nome].score_samples(teste[feats]))
    ax.plot(r, p, color=cor, label=nome, lw=2)
ax.axhline(teste.rotulo_fraco.mean(), color=VERMELHO, ls="--", lw=1, label="aleatório")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision × Recall no teste")
ax.legend()
plt.show()

### Etapa 4 — Sensibilidade ao `contamination` (versão principal)

Mostra o equilíbrio entre alertar mais (Recall sobe) e errar mais (Precision cai). Serve para escolher o limiar de operação.

In [ ]:
VERSAO = "A · principal"
feats = VERSOES[VERSAO]
sens = [{"contamination": c, **avaliar(treinar(feats, c), feats, teste, "teste")} for c in [0.03, 0.05, 0.07, 0.09, 0.12, 0.15]]
display(pd.DataFrame(sens).drop(columns="base").round(3))

### Etapa 5 — Quais regras o modelo pega?

Mostra o Recall **por regra** no teste: das falhas de cada regra, quantas viraram alerta. Isso aponta os tipos de anomalia que o modelo enxerga sem que ninguém tenha dito a ele qual é a regra.

**O que esperar:** o Isolation Forest é bom em valores **extremos**, como desconto acima do preço ou preço negativo. É fraco em valores impossíveis que ficam **colados** nos normais: a quantidade 0 fica a um passo da quantidade 1, que é a mais comum. No teste local, ele pegou só 9% a 17% das quantidades 0. É por isso que **modelo e regras se complementam**: a regra cobre o impossível conhecido, e o modelo cobre o estranho que ninguém previu.

In [ ]:
REGRAS = {"quantidade 0": teste.quantidade < 1,
          "preço ≤ 0": teste.preco_unitario <= 0,
          "desconto > preço": teste.desconto_aplicado > teste.preco_unitario,
          "desconto negativo": teste.desconto_aplicado < 0}

linhas = []
for nome, feats in VERSOES.items():
    alerta = modelos[nome].predict(teste[feats]) == -1
    for regra, mask in REGRAS.items():
        linhas.append({"versão": nome, "regra": regra, "itens com falha": int(mask.sum()),
                       "Recall": alerta[mask.values].mean() if mask.any() else np.nan})
display(pd.DataFrame(linhas).pivot(index="regra", columns="versão", values="Recall").round(3).reset_index())

### Etapa 6 — Alertas que nenhuma regra marcou

É o ganho potencial do P1: itens aprovados em todas as regras que o modelo achou estranhos. Para cada um, a coluna `motivo_provavel` traz a feature mais distante do padrão do treino, medida em z-score robusto (mediana e desvio absoluto mediano).

Esses itens **não são erros automaticamente**. Cada um pode ser uma anomalia real que a regra não cobre, ou um caso legítimo e raro, como um produto caro ou um kit. É o tipo de lista que se leva ao time de negócio para validar.

In [ ]:
modelo, feats = modelos[VERSAO], VERSOES[VERSAO]
t = teste.copy()
t["score_anomalia"] = -modelo.score_samples(t[feats])
t["alerta"] = (modelo.predict(t[feats]) == -1).astype(int)

mediana = treino[feats].median()
mad = (treino[feats] - mediana).abs().median().replace(0, np.nan)
mad = mad.fillna((treino[feats] - mediana).abs().mean()).replace(0, 1)
z = ((t[feats] - mediana) / (1.4826 * mad)).abs()
t["motivo_provavel"] = z.idxmax(axis=1)
t["z_motivo"] = z.max(axis=1).round(1)

sem_regra = t[(t.alerta == 1) & (t.rotulo_fraco == 0)].sort_values("score_anomalia", ascending=False)
print(f"Alertas no teste: {t.alerta.sum():,} | com rótulo: {((t.alerta == 1) & (t.rotulo_fraco == 1)).sum():,} | "
      f"sem regra: {len(sem_regra):,}")
print("\nMotivo provável dos alertas sem regra:")
display(sem_regra.motivo_provavel.value_counts().rename_axis("motivo_provavel").reset_index(name="alertas"))
display(sem_regra[["id_item_pedido", "id_pedido", "sku", "quantidade", "preco_unitario", "desconto_aplicado",
                   "n_itens_pedido", "valor_liquido_pedido", "score_anomalia", "motivo_provavel", "z_motivo"]].head(25))

### Etapa 7 — Simulação do streaming: pontuar micro-lote a micro-lote

O modelo treinado (até 07/10) pontua cada lote do teste **separadamente**, como aconteceria a cada chegada no Data Lake. O % alertado por lote precisa ficar estável: um salto indica um lote com problema, ou uma mudança nos dados que pede retreino.

In [ ]:
def pontuar_lote(df_lote, modelo=modelos[VERSAO], features=VERSOES[VERSAO]):
    """Recebe as features de um micro-lote e devolve score e alerta por item."""
    out = df_lote[["id_item_pedido", "lote"]].copy()
    out["score_anomalia"] = -modelo.score_samples(df_lote[features])
    out["alerta"] = (modelo.predict(df_lote[features]) == -1).astype(int)
    return out

por_lote = []
for lote, df_lote in teste.groupby("lote"):
    s = pontuar_lote(df_lote)
    por_lote.append({"lote": lote, "itens": len(s), "alertados": int(s.alerta.sum()),
                     "% alertados": round(s.alerta.mean() * 100, 2),
                     "% rótulo fraco": round(df_lote.rotulo_fraco.mean() * 100, 2)})
por_lote = pd.DataFrame(por_lote)
display(por_lote)
# Média ponderada pelo nº de itens: lotes muito pequenos (1 item → 0% ou 100%) não distorcem
grandes = por_lote[por_lote.itens >= 50]
print(f"% alertados no teste: {por_lote.alertados.sum() / por_lote.itens.sum():.2%} | "
      f"lotes com 50+ itens: mín {grandes['% alertados'].min():.2f}% | máx {grandes['% alertados'].max():.2f}%")

### Etapa 8 — Gravação dos scores

Grava o score e o alerta de **todos os itens** (treino e teste), pontuados pela versão principal. A tabela é derivada e recalculada a cada execução (`overwrite`), e o caminho fica dentro de `squad1/dupla1/ml/`. Pode alimentar uma página de alertas no Looker ou a comparação com o Autoencoder.

In [ ]:
import json
from datetime import datetime

modelo, feats = modelos[VERSAO], VERSOES[VERSAO]
scores = df[["id_item_pedido", "id_pedido", "sku", "lote", "data_lote", "split", "rotulo_fraco",
             "quantidade", "preco_unitario", "desconto_aplicado"]].copy()
scores["score_anomalia"] = -modelo.score_samples(df[feats])
scores["alerta"] = (modelo.predict(df[feats]) == -1).astype(int)
scores["limiar"] = -modelo.offset_
scores["versao_modelo"] = f"isolation_forest | {VERSAO} | contamination={CONTAMINATION}"
scores["pontuado_em"] = datetime.now()

(spark.createDataFrame(scores).write.format("delta").options(**adls_options)
    .mode("overwrite").option("overwriteSchema", "true").save(caminhos["scores_itens"]))
print(f"[OK] {len(scores):,} scores gravados em {caminhos['scores_itens']}")

metricas_teste = resultado[(resultado["versão"] == VERSAO) & (resultado.base == "teste")].iloc[0]
dbutils.notebook.exit(json.dumps({"versao": VERSAO, "contamination": CONTAMINATION,
                                  **{k: round(float(metricas_teste[k]), 4)
                                     for k in ["% alertados", "Precision", "Recall", "F1", "PR-AUC"]}}))

### Etapa 9 — Conclusões

> Execução de 08/11: treino com lotes até 07/10 (8.075 itens) e teste com 08/10 e 15/10 (6.121 itens, 8,35% rotulados). PR-AUC de um modelo aleatório: 0,083.

**Resultado no teste**
| Versão | Features | % alertados | Precision | Recall | F1 | PR-AUC |
|---|---|---|---|---|---|---|
| **A · principal** | 7 | 6,5% | **0,754** | 0,587 | 0,660 | 0,741 |
| A · só item | 4 | 8,1% | 0,693 | **0,675** | **0,684** | **0,745** |
| A · sem valor | 6 | 6,2% | 0,703 | 0,519 | 0,597 | 0,602 |
| A · 9 features | 9 | 7,0% | 0,616 | 0,519 | 0,563 | 0,548 |
| A · + rastreio | 17 | 10,1% | 0,424 | 0,515 | 0,465 | 0,334 |

**Recall por regra no teste**
| Regra | Principal | Só item | + rastreio |
|---|---|---|---|
| Desconto > preço | 100% | 100% | 90% |
| Desconto negativo | 75% | 100% | 50% |
| Preço ≤ 0 | 56% | 100% | 39% |
| Quantidade 0 | 14% | 17% | 10% |

**Conclusões**
1. **O modelo funciona sem rótulo:** a versão principal tem PR-AUC de 0,74, quase 9 vezes o aleatório (0,083). Três em cada quatro alertas (Precision 0,75) são itens que as regras também reprovam.
2. **Versão escolhida: A · principal (7 features).** Ela tem a maior Precision e o menor volume de alertas sem regra (98). A "só item" tem F1 e Recall um pouco maiores, mas alerta mais (8,1%) e gera mais alertas sem regra, o que dá mais trabalho de revisão. Se o objetivo for cobertura máxima, ela é a alternativa.
3. **O rastreamento atrapalha:** com as 10 features de rastreio, o PR-AUC cai de 0,74 para 0,33 e os alertas sobem para 10%. Isso confirma a EDA: os tempos de entrega não têm relação com erro de item, e 89% dos pedidos têm 1 evento só. O modelo passa a alertar pedidos com entrega incomum em vez de itens com valores quebrados. **Anomalias de entrega pedem um modelo próprio**, por pedido.
4. **O valor líquido é a "pista" mais forte:** sem ele, o PR-AUC cai de 0,74 para 0,60. Ainda assim, o modelo continua muito acima do aleatório, então ele não depende só dessa feature.
5. **Ponto cego: quantidade 0.** O modelo pega só cerca de 14% delas. O Isolation Forest isola **extremos**, e o zero fica colado no 1, que é a quantidade mais comum. **Modelo e regras se complementam:** a regra cobre o impossível conhecido, e o modelo cobre o estranho que ninguém previu.
6. **Ganho sobre as regras:** dos 398 alertas no teste, **98 não têm nenhuma regra quebrada**. Entre eles há preços claramente quebrados que passam por todas as regras:
   - ração Pedigree 10 kg a R$ 5.292;
   - ração Golden 10 kg a R$ 3.367;
   - Ariel 1 kg a R$ 1.690,50;
   - esponja Bombril a R$ 1.388,50;
   - molho Pomarola a R$ 1.272.

   Também aparecem casos provavelmente legítimos, como refrigerantes em quantidade 12 (kits). A lista vai para validação de negócio e sugere uma **regra nova na etapa 1**: o preço do item muito acima do preço típico do SKU.
7. **Estável no streaming:** pontuado lote a lote, o % alertado fica entre 5,5% e 7,7% nos lotes com 50 itens ou mais. Não há sinal de mudança nos dados que peça retreino.

**Limitações**
- A versão principal foi escolhida olhando o teste, então as métricas dela ficam levemente otimistas.
- O rótulo fraco não é a verdade: um "falso positivo" contra o rótulo pode ser uma anomalia real que a regra não cobre (item 6).
- O teste tem só 2 dias de lote. O ideal é reavaliar quando chegarem lotes novos.

**Para a comparação com o Autoencoder (Guilherme & Carlos):** usar o mesmo período de teste (lotes de 08/10 em diante), as mesmas métricas e o mesmo rótulo fraco. Também vale comparar quantos dos 98 alertas sem regra o Autoencoder também encontra.